In [21]:
import requests
import pandas as pd

In [2]:
API_KEY = "vfkQQ0xCxtZG6pRYnhmTeACfZw1xsm63AXY8N3XM"
SEARCH_URL = "https://api.nal.usda.gov/fdc/v1/foods/search"
NUTRIENT_MAP = {
    "protein": "Protein",
    "calories": "Energy",
    "fiber": "Fiber, total dietary",
    "sodium": "Sodium, Na",
    "fat": "Total lipid (fat)",
    "carbs": "Carbohydrate, by difference"
}

In [3]:
def get_nutrient_value(food, nutrient_name):
    food_nutrients = food.get("foodNutrients", [])
    for nutrient in food_nutrients:
        if nutrient.get("nutrientName") == nutrient_name:
            return nutrient.get("value", None)
    return None

In [16]:
def search_specific_food(input):
  parameters = {"query": input,"api_key": API_KEY}
  response = requests.get('https://api.nal.usda.gov/fdc/v1/foods/search',params = parameters)
  response = response.json()
  data_count = 0
  for food in response.get('foods', []):
        if food.get('dataType') == 'Branded':
            continue
        print(f"{data_count+1})")
        print("This food is a",food.get('dataType'))
        print(food.get('description'))
        #print("Score: ",food.get('score'))

        if food.get('ingredients') is not None:
          print("Ingredients: ", food.get('ingredients'))

        for nutrient in food.get('foodNutrients', []):

            if nutrient.get('nutrientName')in NUTRIENT_MAP.values():
              if food.get('dataType') == 'SR Legacy' and nutrient.get('nutrientName') == 'Energy' and nutrient.get('unitName') == 'kJ':
                    continue
              print("Name:",nutrient.get('nutrientName'), "  Value: ",nutrient.get('value')," "+nutrient.get('unitName'))



        print()
        data_count+=1
        if data_count >=8:
          break

search_specific_food('banana')


1)
This food is a SR Legacy
Bananas, dehydrated, or banana powder
Name: Protein   Value:  3.89  G
Name: Sodium, Na   Value:  3.0  MG
Name: Total lipid (fat)   Value:  1.81  G
Name: Carbohydrate, by difference   Value:  88.3  G
Name: Energy   Value:  346  KCAL
Name: Fiber, total dietary   Value:  9.9  G

2)
This food is a Survey (FNDDS)
Banana, baked
Name: Protein   Value:  0.82  G
Name: Total lipid (fat)   Value:  3.21  G
Name: Carbohydrate, by difference   Value:  32.43  G
Name: Energy   Value:  161  KCAL
Name: Fiber, total dietary   Value:  1.8  G
Name: Sodium, Na   Value:  3  MG

3)
This food is a Survey (FNDDS)
Banana, raw
Name: Protein   Value:  0.74  G
Name: Total lipid (fat)   Value:  0.28  G
Name: Carbohydrate, by difference   Value:  22.71  G
Name: Energy   Value:  97  KCAL
Name: Fiber, total dietary   Value:  1.7  G
Name: Sodium, Na   Value:  0  MG

4)
This food is a Survey (FNDDS)
Banana chips
Name: Protein   Value:  2.3  G
Name: Total lipid (fat)   Value:  33.6  G
Name: Car

In [17]:
def get_food(input):
    p={"query":input,"api_key":API_KEY}
    response =requests.get("https://api.nal.usda.gov/fdc/v1/foods/search", params=p)
    response =response.json()
    return response.get("foods",[])

In [20]:
def rank_foods(food_name,nutrient_key, topn=5, highest=True):
    if nutrient_key not in NUTRIENT_MAP:
        print("Sorry, we do not have this information!.")
        return
    nutrient=NUTRIENT_MAP[nutrient_key]
    foods = get_food(food_name)
    ranked_list =[]
    for food in foods:
        if food.get("dataType") =="Branded":
            continue
        description = food.get("description","Unknown Food")
        data_type = food.get("dataType","Unknown Type")
        nutrient_value = get_nutrient_value(food,nutrient)
        if nutrient_value is not None:
            ranked_list.append({"Food": description,"Type": data_type, nutrient_key.capitalize(): nutrient_value})
    ranked_list = sorted(ranked_list,key=lambda x: x[nutrient_key.capitalize()],reverse=highest)
    df =pd.DataFrame(ranked_list[:topn])
    return df
rank_foods("pasta", "fiber", topn=5, highest=True)

,Food,Type,Fiber
0,"Pasta, vegetable, cooked",Survey (FNDDS),4.3
1,"Pasta, whole grain, cooked",Survey (FNDDS),3.9
2,"Pasta, dry, enriched",SR Legacy,3.2
3,"Pasta, dry, unenriched",SR Legacy,3.2
4,"Pasta with sauce, meatless, school lunch",Survey (FNDDS),2.9
